In [1]:
import os

from substrate.runtimes.document_intelligence.service.pipeline import ExtractionPipeline

from pdfqa_rag.config import settings

pipeline = ExtractionPipeline(ocr_size='tiny', device='gpu:0')
data = open(settings.ROOT_DIR / 'data/documents/PaperText/brochure.pdf', 'rb').read()
result = pipeline.extract(data, 'brochure.pdf')

for page in result.pages:
    print(f'--- page {page.page_number} ---')
    print(page.text[:300])
    print(f'[{len(page.images)} image(s) extracted]')
    print()

os.makedirs('brochure_out/imgs', exist_ok=True)
markdown = result.markdown
for page in result.pages:
    for img in page.images:
        fname = f'imgs/{img.id}.png'
        with open(f'brochure_out/{fname}', 'wb') as f:
            f.write(img.data)
        markdown = markdown.replace(f'cid:{img.id}', fname)

with open('brochure_out/brochure.md', 'w') as f:
    f.write(markdown)
print('written to brochure_out/brochure.md + brochure_out/imgs/')

Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/ravikumar/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
/home/ravikumar/Projects/agent-framework/pdfqa-rag/.venv/lib/python3.13/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/ravikumar/.paddlex/official_models/UVDoc`.
Creating model: ('PP-DocBlockLayout', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/ravikumar/.paddlex/official_models/PP-DocBlockLayout`.
Creating mo

--- page 1 ---
Creating PDF/UA

An assertion of PDF/UA conformance places stringent demands on both document authoring software and the author responsible for the conformance status of a particular document. In PDF, text strings of arbitrary lengths may occur in almost any content stream for a wide variety of more
[1 image(s) extracted]

--- page 2 ---
Nhat is PDF/UA?

PDF/UA is the common name for ISO 14289. An international standard approved in 2012, PDF/UA defines how to represent electronic documents in the PDF format in a manner that allows the file to be accessible. The standard identifies PDF components and properties relevant to this objec
[1 image(s) extracted]

written to brochure_out/brochure.md + brochure_out/imgs/


In [5]:
from substrate.capabilities.knowledge.chunking import StructureAwareChunker

markdown = open('brochure_out/brochure.md').read()

chunker = StructureAwareChunker(chunk_size=1800, overlap=250)
chunks = chunker.chunk(markdown, metadata={'source': 'brochure.pdf'})

print(f'{len(chunks)} chunks')
for c in chunks[:40]:
    text = c.content[0].text
    print(f'--- chunk {c.metadata["chunk_index"]} (section {c.metadata["section_id"]}) ---')
    print(text)
    print()

11 chunks
--- chunk 0 (section 0) ---
An assertion of PDF/UA conformance places stringent demands on both document authoring software and the author responsible for the conformance status of a particular document. In PDF, text strings of arbitrary lengths may occur in almost any content stream for a wide variety of more-orless unfortunate reasons. Developers can take very little for granted, but PDF/UA ensures a focus on what matters for accessibility purposes. PDF/UA makes it possible to deliver accessible content with the reliability that is a hallmark of PDF.To comply with PDF/UA, all content on the PDF page must be correctly categorized as artifact or real content. All real content must then be correctly characterized in semantic terms and a tags tree provided, to indicate the logical reading order of the document. High quality input PDF files are required, including several specific technical requirements for font embedding and character encoding, among other requirements.

--- ch

In [11]:
from dataclasses import replace
from substrate.kernel.core.content import ImageBlock
from substrate.kernel.storage.vector import Document
from substrate.runtimes.embedding_reranker.service.embedding import EmbeddingReranker
from substrate.agents.storage import InMemoryVectorStore

async def embed_all():
    er = EmbeddingReranker(
        embed_server_url='http://localhost:8031',
        rerank_server_url='http://localhost:8032',
    )

    # Text: attach the embedding to the SAME Document the chunker built —
    # keeps content + chunk_index/section_id/source metadata intact.
    text_docs = []
    for c in chunks:
        vec = await er.embed_text(c.content[0].text)
        text_docs.append(replace(c, embedding=vec))

    # Images: build real Documents from the extraction result (still has
    # page_number/confidence) — not from re-read filenames, so provenance
    # survives into the vector store.
    image_docs = []
    for page in result.pages:
        for img in page.images:
            vec = await er.embed_image(img.data)
            image_docs.append(
                Document(
                    content=[ImageBlock(data=img.data, media_type="image/png")],
                    embedding=vec,
                    metadata={
                        "source": "brochure.pdf",
                        "page_number": page.page_number,
                        "image_id": img.id,
                        "confidence": img.confidence,
                    },
                )
            )

    await er.aclose()
    return text_docs, image_docs

text_docs, image_docs = await embed_all()
print(f'{len(text_docs)} text docs, {len(image_docs)} image docs')

# Store both into a lightweight, dependency-free in-memory vector store —
# same VectorStore Protocol as PgVectorStore, so this is a drop-in stand-in
# for dev; swap to PgVectorStore later with no other code changes.
store = InMemoryVectorStore()
await store.add(text_docs, collection="brochure")
await store.add(image_docs, collection="brochure")

# Sanity check: search using the first text doc's own embedding as the query —
# it should come back as the top hit.
results = await store.search(text_docs[0].embedding, collection="brochure", limit=3)
for r in results:
    print(round(r.score, 3), r.metadata, r.to_text()[:80])

11 text docs, 2 image docs
1.0 {'source': 'brochure.pdf', 'chunk_index': 0, 'section_id': 0, 'prev_chunk_id': None, 'next_chunk_id': 'c09fcd4e-e6bc-4ebe-84bf-05d144ba418f'} An assertion of PDF/UA conformance places stringent demands on both document aut
0.834 {'source': 'brochure.pdf', 'chunk_index': 5, 'section_id': 5, 'prev_chunk_id': '3ba1e25c-d12a-474c-a4b3-2117f3213bf7', 'next_chunk_id': '5caeef05-fc2f-4cdf-9bcc-d399d21ab7a8'} PDF/UA is the common name for ISO 14289. An international standard approved in 2
0.833 {'source': 'brochure.pdf', 'chunk_index': 1, 'section_id': 1, 'prev_chunk_id': '201ccb26-461f-4879-b77c-d4df8ed01140', 'next_chunk_id': '649520ce-c5b8-4fdd-b423-9ef1f5ef5333'} Conformance with ISO 14289 enables a top-quality reading and navigating experien
